# QSV2G — Stage 03: Quantum Noise and QSS Reliability

**Purpose.** Extend the previously executed **single-share qutrit X-error** experiment with explicitly defined **dephasing**, **depolarizing**, **flagged share erasure**, and a **combined** channel. Produce exact density-matrix diagnostics, seeded finite-shot Monte Carlo measurements, uncertainty intervals, original-baseline comparisons, and professor-facing CSVs/plots.

**Protocol under study:** resource-indexed $(2,3)$ three-qutrit encoding of classical trits as used in the referenced QSV2G Section IV (the included snapshot is verified by SHA-256).

**Run:** Kaggle Notebook → **Accelerator: None (CPU)** → Run All. NumPy, pandas and Matplotlib are sufficient; neither Qiskit nor `pqcrypto` is needed for this quantum-subsystem-only experiment. The output is `qsv2g_stage03_quantum_noise_results.zip` in Kaggle Outputs.

**Research scope:** ideal noiseless decoding plus **simulated** noise channels, not a physical quantum network, authenticated quantum distribution, a malicious-secure VQSS construction, or a proof of full-transaction security. Experiments preserve known limitations from Stages 01–02. The saved legacy X error CSV is a historical comparison, not reclassified as a Stage 03 run.

## 1. Model and limitations

For $s\in\mathbb{Z}_3$, the QSV2G classical-trit encoding is

$$|\Psi_s\rangle=\frac1{\sqrt3}\sum_{r=0}^2 |r,\;r+s,\;r+2s\rangle,$$

with all arithmetic mod 3. Any two shares at indices $i<j$ recover $s=(x_j-x_i)(j-i)^{-1}\pmod3$ from **computational-basis** measurement. This models recovery of a classical trit; it does **not** claim arbitrary unknown quantum-secret reconstruction.

Channels act **only on the SECOND share of the selected authorized pair**, matching the previous QSV2G X-noise convention. Under the defined stochastic channels $0\le p\le 0.30$:

- **X:** $\mathcal X_p(\rho)=(1-p)\rho+p X_j\rho X_j^\dagger$, with $X|k\rangle=|k+1\rangle$.
- **Dephasing:** $\mathcal D_p(\rho)=(1-p)\rho+p\sum_{k=0}^2 P_{jk}\rho P_{jk}$. Coherence can decline **without** changing the computational-basis classical outcome distribution.
- **Depolarizing:** $\mathcal P_p(\rho)=(1-p)\rho+\frac p9\sum_{a,b=0}^2 (X_j^aZ_j^b)\rho (X_j^aZ_j^b)^\dagger$.
- **Flagged erasure:** with probability $p$, the selected share is lost and the selected-pair attempt **aborts**, without inventing a qutrit outcome. With probability $1-p$, the original state survives. Loss is modeled by an orthogonal erasure flag outside the 27D state space.
- **Combined:** $\mathcal D_p\!\circ\mathcal P_p\!\circ\mathcal X_p$ on that share. The common $p$ is a **channel parameter**, not a measured physical link error rate.

**Definitions of metrics.** `correct_rate` = correct decoded trits / all attempts (erased attempts counted unsuccessful); `wrong_rate` = wrong decoded trits / all attempts; `abort_rate` = flagged erasures / all attempts. Density-matrix **state fidelity** is $\langle\Psi_s|\rho|\Psi_s\rangle$. Erasure has a **conditional surviving-branch fidelity** of 1 and an explicitly named **orthogonal-erasure extended-space fidelity** of $1-p$ (these must not be conflated). `codespace_probability` is the overlap with span$\{|\Psi_0\rangle,|\Psi_1\rangle,|\Psi_2\rangle\}$; it is NOT proof of correct logical value.

There is no full authorization-token generation in this stage. `diagnostic_mismatch` means the recovered trit differs from the known encoded $s$ in a controlled experiment; it is not a standalone demonstration that a production commitment, authority attestation, or VQSS check rejects all attacks.

In [ ]:
from pathlib import Path
import os, sys, json, hashlib, zipfile, gzip, csv, platform, io, zlib, base64, ast
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from importlib.metadata import version, PackageNotFoundError

IN_KAGGLE = Path('/kaggle/working').is_dir()
OUT = (Path('/kaggle/working/qsv2g_stage03') if IN_KAGGLE else Path.cwd() / 'qsv2g_stage03_outputs')
OUT.mkdir(parents=True, exist_ok=True)
ZIP_PATH = (Path('/kaggle/working') if IN_KAGGLE else Path.cwd()) / 'qsv2g_stage03_quantum_noise_results.zip'
LOG=[]
def report(*items):
    msg=' '.join(str(x) for x in items)
    LOG.append(msg); print(msg)
def digest(data):return hashlib.sha256(data).hexdigest()
def library_version(name):
    try:return version(name)
    except PackageNotFoundError:return None
SEED=20261027
N_SHOTS=1000  # per (noise model, authority pair, secret symbol, error probability)
ERROR_RATES=[0.00,0.01,0.03,0.05,0.10,0.20,0.30]
PAIRS=['AB','AC','BC']
MODELS=['x','dephasing','depolarizing','erasure','combined']
SYMBOLS=[0,1,2]
report('QSV2G Stage 03 / Kaggle runtime:',IN_KAGGLE,'/ output:',OUT)

## 2. Verify embedded original source and baseline files (offline-capable)

In [ ]:
CORE_Z_B64='eNrdPf1z27aSv/uv4LXzhqQNybbc5uWUshPXUVtf81XbycvVo+NQJCwxokiGpGy5ef7fb3fxQfBDtpt05mYubWKJABaLxX5jQX/zzTe/n78f/WKd87CKs9Q6fW8V/IoXPA25lRdZlVW3Obecgpc8KMKFVcardRJgX2a9fnNhRTxPslseWSUP1wW3ytuy4it3uHOerYuQjy2C76+COH3qy1lO3/tn/DoueTSs+Ga48/s6SKv1amzxTRBW1uifL6xqUXA++LSuiriyyiqo+DWMzYrSuokrwCJY5QlMmgdxYa14UMLUK55W5XDn7e8nYwvArIPEevVy8Nvk1eCfT55a+/jlxfnx4Mn31nUcWPmnsLjNq8x6c2a9yXl6fv7S+tGzjobfD3cuFnFpwf9pVlmrIInDOFuXA7nAa17EV3EwSzgs7XxoHa+rBcwch4BjZIWLIE15UjJFD54WWZIgbmznk1gnILzKilsrSCOruskGAUDIYJ231iJLeQk/AySkJrQVlOV6leNHWN8333yzc1VkK8v3r9YVTOH7VrzKs6ICgIAxDSl35KNZUPIn36lvi6BcJPFMf10Fofr8scxS9XkVVAv1OdOgYEEFr+qvi3UVJ/rbegbcEvJStwMX5FdxwvX3eMUF4lFQBWECi+Klwlw/YtZVzJNIdMwBD0BXdXqLaFED8GScztXz4/RWTZICnYB8sHW56Ck2eV4E+eJ2uAj+hKUN8yJexVV8zcthGOcLXpTDgAeRBjc5/+Xk1c7Oy+OfJi8tz7KJgwdnk58nZ5PXJ5PB6GD0xN45fnfx65uz04vTyTl0cuxjm9k/wd8T2905fwmS4Vmf4eH4AB+PD7FhPLrb2aFlWm9RtMIsOZ7BlM5kE3LaX3e8Y8GfHLrs7ET8CoTx0zouuBNmaRQLqeObPAnSwOgeXxGr6j7iKf4pApCy1mTmeDEJ7r2fzT46WQ0wLuMU5A7UgJMxa3Zb8dIdAzrAcSkszPdRL/i+PbapzWaWLXjNHoufw9mT70CLZBEAcIcRp092UIZxbLt3/dNUa5DqLdNQG04TA2eV9vhSY71xraussDZWnFrZdAvoJC6rGvK2wf1jozg0xn4uq8JZumMN4lqAWLJrBFICjXnkZEPC03G3rdUBMCwGtTDLssR1LYCQodJ5DTpAz5Xt1Lt4AZSYFEVWOFf2u7Rc5znNJATG+oyEQkr7fhqsgGR3Y+sqyWCb0/kgz2Ai6zpI1iByqFzgK+qsLA2S5BbWV8LP7IZHtuSIMEizFGgj2EEigwseRiBhpWOQj9GK/SW/Lb2LYs1R8+VBEaCy9hwQCAabB714ikraJwbwfg6SkrtDyR/2uroaPFVzw+CS+wKDIriROAhhKAEq13h16brpbLQacys26Ra3aDPdPlzstW6XfTbDOa+cmh1dz5N8T1q8hMYNPDMYlilxuGsCM+hZy4mUjs2lGjRlsFkxaEVONHUfg48QkO34CLnZjg6Nd3rpdSkHT5uIKIFYjtWopihslAzc7bSGbEy+UoOJv4BloxL3XasMyRyuZA/JM7vE9qXcKjCKP8cbEIasiMA+p3PYSF7xYhWnwA5xyNBk8GiQ8HReLQaSQ6E/QYP+QzSrCClbVx7ubFAUwa0j1otLusIFiTlrCoaLdbr0BKte1bQBGNaeZ8FkDvVwh1XmE7s43wFfxHPb3aMGkwqiHYbKdf5aL1F1keZ7WC6C0fdPnBYlQMnGcyAlII3jwbY7S2Z1gUDDMOU32NiCAMpVgLZbsMLK55+cgM1aYMJsBYTkvuyMHaj/Yhld+XxTFeCHOaBbKmbFy1UPDrpt69QSVA5s7eQF4BynV6A9U+9oJPeedsy2n1n0Q29YjBtWBOmcO4cMPZphyOPESfdh4N6hIeKVp9GhCao9nGJP7MdlPO0lyzOcds+rzB2EJ5fjdCq2T3T0ARMnCWY8gZ0gYaiJ8GurYWfnW2sAfyzpDjfdXmzZOR15aT7kt9wZsQg5Gr/iJiR8czh66u58oA6Cdy8vD9jhlF0esoPplMle7s4f1CWKg7kDTYPDqTsMSjIeusuvDSiHEgp0rcHsQ4fyE3gTI3fnBOKA5oAD/A8GAQL6E/53KD+1UAI/66UB4FD17ZtMEBeMCfcF9/pST4An7c/iSmkE4EVPPaIHRE2cZVmAtMLPJAbrNx8CaxTxxs/B/BXOHyMWuM+3NH4YAX+z05H7HBEWjIbdNNBfR9SK9HhO0wlnrshmwSxOwDXjJS1zVjo00N3dHQnnINp4YJIdosF8FWycxiDXlWwmfEHoFZdhkpW82e0SwEyBtmB6E++QDw5HLviePEksIhF6zFdBnIAVtl2Tbx0Yt78/YvDjH00KgwCnJboZfgXsAn+1QxBmEAfxqKMqkdyebQ8/gscBfspnbB8fPJ3d2SSU+BXlEkFp4TWEdlkL7QFD/Uk7ykaGuAawszOvjwMcJCEOuFyC0NZf9oDH3YZ23vPQhwsCdw9/zmZuc1VDUF0c9A2CQKH+OP6493QKWBCOH9s4os5mT123q801RKDqm1eTX45JfDe5M/q4C5/yeP8IZPYIn2KU6EjhPpLCrfgf93QTl94BSO9RLb0EcXf3YxurIxfstNjGT2XpEyc6hqX8d17G8P1HiFAgrvQL698FK/ZK+Dsqf7T2SdKOXG0RFduVOIODwot8dZqSh2JhgC7ZSUvYn7zISgeWAf+5rbXorS5MhMdi8GXBHMDE/ccR/hzt4qepd6jl/6hBYyFiMkyK1iGPfGQLCKQdWCGzlhBTyVWvYBsiRA0/IC2pB3ZgB+4QHJBFkCPdB4eNGWjYc/oX1Gz60XGHF2JCmXPwMQEhZsNPzCrQ9dj4V8E6qaSXq+l+TkkL6yM540iPtQjWg8RMYVjZFWYFIL4Gy1paFMUG1lFPLmRIgOVkVpDnSYzuvSV6Dj5YHKMFq8qsasGt88nJm9cvwDVMYDDmaxC+NeOwF1zOj36QQpZ+rtm1d4nB7GU+pU3LcdNwpdPmjjfI6sAwlyGnSJlALQWxgFR9NAhVnzsE9nOIs0f1JqgtUDoRKDoMF1kM7vl/stwjWPv0Lw1Xkr1Zs821F8XXqyxChcYkr6CzLCg0tqCDs7kG4/+PI3OXaajiIxJYv7xdzbLEoT2lZh2FpNnKcwRJoPESbOKg/gZWrQXbgQkHm7W7CybEodHAY4AbdtMSCqxVgpsK2gzUqJjoQ2CaKvr54Yh9AHH6IBf2R7fHH0fsD+jxh+wBAlp6l1MtcWVD4rQ+hC3zTD1hNvhXSVB58KGzPWqCIfAd6srPjcjAhgC9ZM1HaVas7DHFpIjxdZRVjpqCqQ/IB2AYW0M3iByaOdAsPnG1AUiaagTvfAieK0gDDbIN7c/HQvvjMdAgXPdJlsALFmEkJid64fVpqZk7UFrfBWsgTMysoczrCe8a2gnJr53GY5HxRAyioIhgbZGVZ2U1UElHMxMG/jMomDhBdXMFof8sCJfSxZTpqd9PqHsde/s+hFKV7zugQK4M7sGvw2WcRh4mLmqnurhtBpoioSeTrsMlX6l02yrx4Zv/zydP7+lfxvPUGBCVgf/k+0Z/gQdfeTW8Z+IhjPXuG4O42zodjImxa25mjvfqzLGth3PK2FnOKeFEaRn2KovWCX+dVT9n6zSiZ61MAt9wT+RNhzeLOFw4dgbCU5aJ3YnwoWs3UgefovLqVOuwWKfOJfRkNmY9IMYfwNIHwG+YTl5gwM7CIKckMTgpYG9EhqYC/6Ynr0AkAWLdOwHuRIAQv2oaWKBdk9gmlxDWNiyrCINnzGHYNdUprRbPZWuXLK29VOn8o+H3e1s20nLqRbr2doByezwgQXuD9IQ6addNrFDe7mydYu5bpO7sM944l9ivUVqnwTW453iyMLQmKZ0w/BbM5/DjNK14kXJBmDxGKwyCDo69FhDAwCLRjUPUAutZWcXVGpztmxi6zbi1xtMWyWYkz3JpPvRPSKiZBbHHmk5RMMqGbfQxf1+SYDNL7Kx8BGa3Xi6dyKh0//CCo0gExe0LcBrRTQE9V/CreOPZn8rr0dwHnH3bxST9VZZEvGjSrciyysNMvyNam7yDyhHTmwwPDMgfNRDFFOrnO1clncaWg8D2sb87vAFW5TIJg4ObcGHl4OmYWz7du8TYgCA415eHY4ghWtnCawY9XNqTa+DNoKhKJIVjP4flcXD9LIQgU2KUENMEnjZmz9sih+gwS7C72fb29O2EnoPV6j4PbyIP8WV01oJDjw46egUiDbIeGDE9jmOVOBkGJA4tZBsKJ3k0tuy9fCiwamXtGJlXiAgLnidBCIHn5fj7g4P+PKKTSwln1u6loPyG/IFI7puZqje4cVpzNWr+Jb+dQ1TWslOmxDY2S4mQ3dLV+XrG8iIGl1GZlqGETHywCMqggu1VjcwWrXrr1SB4xjHXiHhRtOC6z9SK1Rx64m9hBbeWHEJHfyjw61kShwNskTk1bBH+O4gmiR5JnBUpkftS2UT6CdmMmtQol0IoI3ffLpfDnK/sZ1ZuPMzFw+agliEx5YvZsMoc1oRGRZsS+GKYBWiBfbZRzJxyCY5QuODhUhiXHoPj/oXJ1cxxqsHDN6A0TWjMm//VedXWLhucC1LbfKBZFrz+IC+FBob5v4ZlVVysWE+CRqj9LCs6dFgWHiN8Gqgn8FlYsbL0qFfTelwSOddVgpQjmLYgJlIXaSyI/TxfatLaz0PhTtChMn4v0Y2wPgMoewxD75hjUxd4biRv5BIJFU1C0Dk1CUGcWOMI56vJKMEryP2kFJ06pITHgpRysEHOR9CSgDZIWC4V126h4GcQTyAgTof0G4cVUhKaWJeKBgnBvfq7FKdJPvTatmpNaOzVmjioqzW/VKXhyv7vVZr0ZP8/abRtCg0jBEMWV+X87+Qmgq4Bm+KEkcsD8oSDt4oTwDPUE3RtSRO2j+Efkido7ROoeK7JQHVCt1qvI7qI49fQohNUi6lLzDVqAsl5zSm3nfiKoSp+Qfe5foJs0IpdKeLVpSr6HIuynOi6yJTwuYoQyX9sBvLW4XDDrPdYiCBawZEhNx/JVVLh0tbFPrS5YuH3WZ7GPgNhUIGYey1NT73VjDZ6DP/c9YpEg0iSQKbz3KTRzs5zXeck8ywXRXB1FYdioYDrmIRJ5FZ5GvlhtsbQbxynlXcgH/N0XPLKozNbBzgNc5v+VUC6D9igargWZHwED1YCOAuisSznMdhXpviJiYyJfxjt7v7nk8Ehs48n54NfTl5ZsgFWuwjWJTKCkbtIM4iLvA6U+vj7cCTPv1Vuxui15x3WB+uVJ8qwlIW9dbVXQpPAcmAhHQH8bFOrPRadbFHfhYkIsoSmx2BQZkmwxq1lwONLCW7aoRSe+1Cb63mHI3LT6SsVYsVKX8FegWewzhOqDkQBWwUJxDArHlnHk+MXYoxBQGR6K+9ZvEJYLh5RM9Y2bdCiLalo+7CkqKcWzCYsgrqEEbvrQ0Ihu7yZ68NFDYMokstv70Be9bD57+fnZ7wUBaE0IF2PwRTJ7QgSzLiL6sIx1t/UZwvG9zBbreKKoudWJwTl2e9evzufvBDWOsnC5ZjqWf6Nmq1OTcoqIl5cC2ZnQcjQvdgqC3TAoYEzG1aCAGgpVpCgAbxVELH6BIvwSojxI9vt5RmszKEJgW8E26Aeo1SGWVCIcW6gzlrAjlOdayz0qwHZwNE+m5xPzt4DklK0kAaeA+sTdT1iUoMIiGhVrMOqQQiGB1hVsMp50Ti+2koZPa0sPFIzd6bWtANSCT5bxeUqqMJFg1TqYPlzML6c1nQwiHN3T5IZugt+ao3pmk1RppSXVCbCU9iygo4/9NrKSwFp6vYnHMWBU+ccUBNRHQI6gppEHgERqHYsyAVkOnDdXvCaEgoNddCy/YjK/VJa8E1OZ4Ler44sdoWtGpy8eWUzAYERWdI162DV51vIilWqGVKgBQRDhjUAxReaH7GKWvfrYxKpDcDGA8qXeAhYoxVML5f9bOP+46hdYoACWQ+FTZm67rTPzqvZdBuVbyR9DoqUxpM3r8/fvVL6qNujHIYJDyCsQcdJlnrDwuOi4OQJYfLXPBTet4A/8Zs4NUz5jY9ndkoVOS+9wycj5DvSdapMA77CX1RLWGEAHJdhRCo8BmhwsJLZlQrZkxoYjPWSp/6Cb8Bauz0qmkRTdUagM55kN86RyM75NXlfuvcJsCADATNPH+uTyuassLP3QRMcQ+C6TBwwYN0OOPcecHLfDbvldEAwsQImp3Z3tLGDyetD/zidJxyi9LKkrF1VgO0y7ljEWBOAWyx0IgW3WKEdh3EFCg5cLGCL+oZAfStA12d0TueY8LVl3h7sdeThRjPBI/qQweQUzZzgqHvSVQfiqGNAR/lryFH9rITTtAxTPRNaPf0N4x7ji7hmQQEIpQBMkW3Ce+m91LZt7l1OFU6YWr71+TVN4zafhmXjKd/wcI2ajh42wd+A1hJuqvf5DsXyFexZMMebGH/iTYU0TNYRt2q3a78K5qTGwUOewW6he0xO0TN0wagz7m6Mzt7xi2FztoLPBcqm0Akn+WhUL2IuVtDbqbVzceSvAmCX4gGg0A8mtvm1H9OxUpD4mh39g4NDuwk3LOMI9Nm5/2Ly6o0Pzc/U0Rg8/fX47JdJawBES1QMDqKoWGpopJfcB4y6XLiQOq+r6JhixNb6qwMqF8HYa4j/UJ5dKdw5qOTXk38p3H0Q1AqoVSw96g/2+UqFIUYmAzvIEGERlChLIKXr0rPfHp+f28za3Y14BYxcNmKHm60wNa7Av7puwibYEHjSFLaYAmJO+slsngR5yWFr4VmB58oOON43g9Yy3N3Dg4MDrLbSON215jTWDAAMR3COpSdFO+2nzPhnebHl7gevKdTMfncyOCb+hw8/qQERbW+WJmAvJXBxlGF6ece9LHr4xIhhrrHk0MMiZkNcGLibv9isZmNWHLvG4cnR6CcsLbQGP1qHo6cW1j4OyHVWZd4lE7WQgCZp2tHg0xpFV6rVrNjpd2P6yiEFii6zHoMkuDmT94SQvEAWtomihLhecy3RJrAtlGNSCPaePvnu4MC9HB8+mQ7RiBtk/+lBsodll+whcOHJeb0mVAis+Mn9C6QSYF32KLguPn6YVFKqYfzp+cXkrA58SGrtBusxC3yUhJz00s+x6gei3erWo5ppJQlGFCyDQ0u47qguKRbSD0AnPxQczUVwpJFjtpI0cB6LsuqJD79K1PKgANTjPBAXffDuSA/RUr6NBSTxUj98kEkEp5W+zPRUB8/0o1A9anbeJs2T9z7aAMkAwO8GJ0gjI0MPzvSsBiLrGXhewpzrWcz0qt6/sdFXFvbrB5cH0/85nLp79YPDcTfhg5WLbfsyqHH6wTsC/YvaG3wakHTsUeL0PfvsNFcLTlgja6T9GVIZIqskntnbZE6jzv4mEgt1lRdZhhbiI4VwbV7SeFI2qLuox2gVE7capXwbbsCdGsnQ7dnpsByr2eQ2i29qj+W3v7zBYXODQT89vMH1egDt3i0WNHjcFgvM2d9NQ6FsH7XRYWlsdHNx9YB1gNWsV11RBzwhJv758XzYwJM1T6WapJFz9hjiv2NSIQoh3a2WhzrkB1JitcKqtfvME+7N5PXF6cnxRdtChUUGji3EUUGc+s0UrFGnhfBnSVwufBlCbnPUGoanOS2zTdvWsT98yaKlp311o4am7jPDyjp5XexLiMkNsvKlUWOMnNflFqwAmby3WevGG2JR38Haxg8Ek30hTJc6ypobqsbBfTfpZ8oH1SGIEE1ST9ZcmGu8+PsI1yghIB8rLOK88nFNXmthF8ZNOkMT91Ac9QhS5wSCmhbQe9RRD5nvAyQJW4fPtRR1jx8aRN6qU2gqku/WZMw+PjmZvL2wH1IZXwJSLqTOySAH4eWqnlQlMAcF+ZI5ZCVJ1NzHli0vyctHrpIzUaVXpN6/0DuNiLuNBO75KYj9RW8YgHcu27wi2W9Pa/Qu6+TFEjy+7s1OwhfcR2wP+9tD41KBqNK7pauWdP/RuBXavupJ7eqSuLvX3hEzBQK4jUJPAgahkWSYjE7wrBEwG3HdGqrWk9Gkh0MIlPUfnhjFbFFYIu7pVOK8FldQWutUZbeTWyuOBAubpdwA6VqwroHYbyeohQRSZbM1pFYQnhrEag1Ep6u8Chaao8HPp69Pz39F1a62/rGCiwCJ7ZuTIkqPhSwsIqpGxCcuVlvkFqYKJe5ypWiRHzEDQyTvWUDYWADXVPtL4IXn8/AqRGgBzCXP6gV/qHgJ+apugG/9bsD55Pz89M3rpgMgDbqvTTymgTFWDUIIvCLDIoOCYVTbWmtPmamkO+YVlhfGkfGIPg3x+BZlXxHCvAX5rTUBUHjqlRARYPng28AXnQsUb4pA64f35PAQWVU+o99TxLM1ztEIn0Ef/klkxN1GLGUcbeQK8QUNEKv4q5sFhN+YtIIHVxzdaUyse0/wifLm9WofF24rKjNFWsJek7f3UDajK22ORs9+cXouP4vSbo3uj+D+yyNYWJAq0M0wucSL+a3o2GYcWHmWt5J/2M/7bOO1SHtsT15Pzn75b5h4FWxsLPRw9IzAousUYI7t1b8WditFep1EsjS5OhiMdPpHAKgJ6rYQCquNt9UR+cseSb0gppAy5B/nUq8YUJPXzYF2KJU6vvhgt/BA1AwTtJR1TiBnuhwEZ2mUP7SS+Ze25EU/uwLOtqckIGZxh7unn5g1Fb3xpW5vFYu3B6vws9OgItFugxmUgq4Ho8Pja5XbwBXrWtJlu/alceJaD2VIGxm/CBrQq17mBacTpx5pUJsIWP7QSafLtBQ2H06ZIiueVmEyqs35QVh7I8cntsClxYtZWgIeXccOw5c/0A87hyBG8kRgBPygIEGAgDfsseYXOwjltyAk5BC0fCK/3ZkAfM1NoM41N4l3eUCzyzrMCcv4rWZQ9xEcB3Akv8kJWzynn/bzXcsxVLgqPpCjH0aU1atqcgM82WLwpN2CLbiYfLhoBa5qdZqRmIIoAhG5C8LvdNsWorr1zdpIvLhcZHRXtfESLQbLv86WIAAy9zoLIl9xzKPMgUK+ZlVwnUvw1L8m9yrTrXWKtc7GOlfrJJEvwBNRYzGLK6xNHmQpx8JSPE2ssgRsR1q5Xyd+lt0wuD1SqGhlSoB6VZhBS3HqqpKiRl+dFzWf9WXOZEQoerDHCrMrKauyTpbgy7JfMSHSkh+Ig+XnrQFeQ/V0bJCrsWjwRm9YBdIWU2UqHmiKSi3NsaKAS6f1NNM86owzCLE+D0tYgt7uuj5YY4A1EFuqygx47o/eiNlX/IYXVrUIUnqbAN5NjtOwqnHflrB6+/bszft2rsoQ3fpsQYkuLAFkR7zyo4lJLfv4PsdykSWR36k8w1MZLfquLD4TV6ofJ+caZVbjqdd5j8yT1cfaNFWKJ+/w/4cnr/brd2lRrx+oaKC5QGaTnKeaEgahNSo4epuwq3PtbqWhrjJcrcuKbnCmeIezc2KI1DPq7PrP0Ydm0SMaSD22r/auPbS5YY3h+jReFQ2a29dBRgPCEIFMhZbTs8mJ1g2siYBZd7btEBDVy8XZu5NOlhWLBzTz2YLZ1PtpRHFlvW7x+jpVwen1bFDN0MBVa+7TEZlgY7VwVf3+2BNCE/NOLZwsk2wxML6WQ+9ATRBEEM2xp3Ufat4WVbVrbZIXALJtW6Qd+56k2qqc1w6+mt7YIkWLTjFI6pilIli1ByYNL3TUmlAv8K7HD9eQWxV4esLLtXbBGw+1UWs8NK3at8LOm0fPzyy6a61TwiIylsVZ+zOsysC3rNQSL5JB+E4mcXVd12x1T53k5WsijOEWmaQ5mKqbHArlRvFaLcp1fGoWjMlRbcVBrItetNo3UXNCH+neg5yMrj/Iz3e90nfx5rfJ6157IZMBNJXNCBcMorAoBmxUjZvMXGAp5yrIUTEbqx2K91Zibr4WP3w1MbkNWBmUo9vQkEL1sHVuL060g6uKsEgSujD2OEEVi0R9bynt30x5iKMMs5a/h8Ca8JfG42mT3Fjc7jR61o1Tt08aepfVlQyqDJO8AkzVbvd6prsU45p97+lnyJwhaS2vscy7wVuuv2Jawg7r5rAT3LVKju0sl21ZDvvD5ZcUXPdUjcL8hZ3EK/lVZCzacEDHyXbSdsrtssdkF4inJSuP6cfd9lTG2eT87RfkMuRLT4FC7v0pjaZ8KW7/utyGgvJQZqObuRgPDqd7WxMe0IiMYPoZIrFhvuh1W5KjkbB8r8Iu0qsRxwI6EKDkVlxUUYAtpJSIaJiVYj21eotWkuHbMfDAOwlicvu7GllBuSQWnXrqZl1E3mDdmhuteac1NAcLFpZG4XZfJSd77z+0kQDmVmCyvDVJynUbhKSttrBuQ+YnftgHEI+aFUVFDSdhaUFHUVHt8JlaReBV9wGRnRpxFiXh5bsL7om5viQT1RsJW0Fl4anGPavUEj5VNx5lbCn5KJDBqaUFzLyFXEj3zzOWXKuIaY9fVo9pqP/u2lUbrM5T2RQRjOiWUZsz66Yj1WS6eKSuudHtvk341rpYUBmTLHsAqQnphgKmu5dpdmMtMI57ZsXYlmJS5FbVRuC75uRcRnlEV86MF6nUqD+ZMnE3UaTe0bc22iBQOxq1FvvPNhvWTU+nGBwKn7Vn/RCl1e9da++sCEprWN9NWf3l++n2y1169GNveNUDaveNeETa7PLx/vYXOJUNljQNesPFNHHE4jaNHnXvuZjWrITSJrAntGsWRqn6fFAXNIe6Yaeeb0laYJEb7XM7+sOSQHKOGpWkVARMku81YxoquBFTydwEvvjVv6GTKqy8zdKo9I4OWBWHS7/0vn/IazQRI9JFcSRSg+RDCkqaPmOsDnXupVln0kqC7dKxACsZVFZbRWJzT9WSGkeVXgCUricFVbaCwKcJQkTJ4n7pChgSfxvH7Fb9agt6oXddE795sKYUFtJyoc4vjs8u2j5UX25AuX7ppse7MnP7MAe7d477UvqaZD7d1P8SV+uBdH43ld9BjwEIl4oQ8T148npKT9nelxhREmGMGJX9lK8G7eOVb+mNZHNwrIAximzwr4WFB5UQF19nwImt3ztQ0sGyYhx8ywBGzFgxE6xm8XwNk9bmQRym+uubhXfwTB6+4pfaFbmkk9Ip3XgAYa94Xnp1vXGSzbEKw8dFdoqRWxfU6reCS9HeF3Lttq58RpW3ilNHNCo1MKCZd+WI9mu2ouoH7wDXauJj/WgZJB9bM1Buy+ZIYFXhK4i7HlL/7EbV/tGTg4PdQx8WjX+bE9ZE2yNMNRimCTio+7TeKoLL2PMOnzVQ3fOiqvNKQg3hR0/Dba9CsV7d+Ye6M75xiM7KkejIAHhVnUc99RZiNBYJ1ID2ccf79f+Hycm7rvIXI2sGZpYBtzUPvjQtXJYekYMqFqReMXLY9PsE6tsBdH2qWFMO/1E5BI2mTCPQmybpzjheAqRfR9CnmOWlo9PXF5Ozs3dkS+hNdPX04gzHPnnz6u3LyYV56xQTekX0wGFpupZfGg6jGVZ3bvl27ylBKBb5QSVKGEw9wyx7hb+LAZrX/1qIdsHdrS0giXbde8JqIMCrrwirkRQPBNZqjykq+FtUfE9QyxoIYWxUzyp3bGsxtsq5ASVOX3eLik38meQc8aPzpgYxkXzv8Tr1IzDdTuPqaOtkpnOVdHeX7n+qX1rk0SYpEOIHozH4D4x5aVw+pZHit5joh+pFX417/fjCQnlFzdjmYeO2Tn1PxwCrH9pMSCerb+90u4Wl6mbO0q2+buPQLXkC1WGAz/D9XaqgiNW1T0YX/dBmVBHlst43Aph/GiVTDVD1c5s9QVjdWqqexQvU+yjQObSvj+vNLdSnisy8v8vUEb7RVZ3cqk0xj/WNbsbjPrT6zxOJYelYqnGK2F0wPe+Dax7rdE50euDo7HoPrJ4cdTs93YWoUw0tpu1PXXfH9/brZW0Z5zSoqi1c970uO8arZRovkRGvlzGgkCbKhabCy5D/NRGHW+AnlBBy4TvIuNszwf8CWW1HIA=='
TEX_Z_B64='...'
LEGACY_X_Z_B64='eNptkdEKwjAMRd/3LUXSbW2SR7cPGSIDfVJaBT/fNYnS6i4U0nJGcrL76Zrca1lTuqXl7vLl9sguP8/nNef1Wy3p9Fi74+TgAI5Bjz+APXm5E8ftwhyaGDIoEgWJfRNDgiFBEN9EEevjfSFgj+iFwL4QtEvoILGnrY6Mdag7zv+Gc2VIXDrT7ldmiDI+/hrOlWEYCzJSHVbkYwiF0CdTAtxqpGa3UQlTikUah7FO6KZ/palWYhmGhyaGmBLJvvFn3qlWkt4cYh1U5KMkfx4aIiphhr5sDrmZf1TCDAfZATTrx+4N1HeTGQ=='
CORE_SHA256='2c86ca9eecbc5925100722119f061e640df03f993e14909b3078b23406330758'
TEX_SHA256='4ca43b1f6debd2e2bf314ff9adf74f0ad08ba1702e1055eca1943b9533290ef2'
LEGACY_X_SHA256='b55b31f50c15b8cf3d9dea2336bbaef873947041042163763ca9a77fd053bfaa'
core_bytes=zlib.decompress(base64.b64decode(CORE_Z_B64))
tex_bytes=zlib.decompress(base64.b64decode(TEX_Z_B64))
legacy_bytes=zlib.decompress(base64.b64decode(LEGACY_X_Z_B64))
assert digest(core_bytes)==CORE_SHA256
assert digest(tex_bytes)==TEX_SHA256
assert digest(legacy_bytes)==LEGACY_X_SHA256
(OUT/'qsv2g_core_reference.py').write_bytes(core_bytes)
(OUT/'QSV2G_SectionIV_reference.tex').write_bytes(tex_bytes)
(OUT/'legacy_qutrit_x_error_trials_original.csv').write_bytes(legacy_bytes)
legacy_df=pd.read_csv(io.BytesIO(legacy_bytes))
assert len(legacy_df)==21 and set(legacy_df.pair)==set(PAIRS)
assert set(legacy_df.shots)=={900}
report('Core SHA256:',CORE_SHA256)
report('Section IV SHA256:',TEX_SHA256)
report('Legacy X CSV SHA256:',LEGACY_X_SHA256)
report('Legacy X rows:',len(legacy_df),'with',int(legacy_df.shots.iloc[0]),'shots each')

## 3. Extract QSS encoding/decoder from the exact reference implementation

In [ ]:
# Extract only the genuine reference-function ASTs, not imports/PQC initialization.
# This ensures the state and decoder definitions remain byte-traceable to QSV2G.
tree=ast.parse(core_bytes.decode('utf-8'))
names=('qss_state','recover_symbol')
source={n.name:ast.get_source_segment(core_bytes.decode('utf-8'),n) for n in tree.body if isinstance(n,(ast.FunctionDef,ast.AsyncFunctionDef)) and n.name in names}
assert set(source)==set(names)
ns={'np':np,'SLOT':{'A':0,'B':1,'C':2}}
class ProtocolAbort(Exception):pass
def require(v,message):
    if not v:raise ProtocolAbort(message)
ns['require']=require
for name in names:exec(compile(source[name], 'verified_reference_core.py','exec'),ns)
qss_state=ns['qss_state'];recover_symbol=ns['recover_symbol']

I3=np.eye(3,dtype=complex)
X3=np.roll(I3,1,axis=0)
W=np.exp(2j*np.pi/3)
Z3=np.diag([W**j for j in range(3)])
SLOT={'A':0,'B':1,'C':2}
BASIS=np.eye(27,dtype=complex)

def embed_local(matrix,slot):
    mats=[I3,I3,I3];mats[slot]=matrix
    return np.kron(np.kron(mats[0],mats[1]),mats[2])

LOCAL={slot:{'x':embed_local(X3,slot),'projectors':[embed_local(np.diag([int(k==v) for k in range(3)]),slot) for v in range(3)],
     'weyl':[embed_local(np.linalg.matrix_power(X3,a) @ np.linalg.matrix_power(Z3,b),slot) for a in range(3) for b in range(3)]}
       for slot in range(3)}
VECTORS={s:qss_state(s).reshape(-1) for s in SYMBOLS}
CODE_PROJECTOR=sum(np.outer(v,v.conj()) for v in VECTORS.values())
assert np.allclose(CODE_PROJECTOR@CODE_PROJECTOR,CODE_PROJECTOR,atol=1e-12)
for s,v in VECTORS.items():
    assert np.isclose(np.vdot(v,v),1,atol=1e-12)
    assert np.allclose(CODE_PROJECTOR@v,v,atol=1e-12)
    for pair in PAIRS:
        a,b=(SLOT[c] for c in pair)
        prob=(abs(v)**2).reshape(3,3,3)
        for idx in zip(*np.where(prob>1e-12)):
            assert recover_symbol(tuple(pair),idx[a],idx[b])==s
report('Statevector and decoder: verified against exact reference AST, three logical symbols and three authorized pairs.')

## 4. Exact local quantum channels and measurement probabilities

In [ ]:
def from_pure(v):return np.outer(v,v.conj())
def apply_x(rho,slot,p):
    U=LOCAL[slot]['x']
    return (1-p)*rho+p*(U@rho@U.conj().T)
def apply_dephase(rho,slot,p):
    d=sum(P@rho@P for P in LOCAL[slot]['projectors'])
    return (1-p)*rho + p*d
def apply_depolarize(rho,slot,p):
    twirl=sum(U@rho@U.conj().T for U in LOCAL[slot]['weyl'])/9
    return (1-p)*rho+p*twirl
def channel(rho,slot,p,model):
    if model=='x':return apply_x(rho,slot,p)
    if model=='dephasing':return apply_dephase(rho,slot,p)
    if model=='depolarizing':return apply_depolarize(rho,slot,p)
    if model=='combined':return apply_dephase(apply_depolarize(apply_x(rho,slot,p),slot,p),slot,p)
    if model=='erasure':return rho # surviving 27D branch only; erasure flag handled in trial sampling
    raise ValueError(model)
def quantum_validate(rho):
    assert np.allclose(rho,rho.conj().T,atol=2e-12),'Non-Hermitian density matrix'
    assert np.isclose(np.trace(rho),1,atol=2e-12),'Non-TP channel'
    eig=np.linalg.eigvalsh(rho)
    assert eig.min()>-3e-12,'Density not positive'

def pair_measure_probabilities(rho,pair):
    # qutrit layout A,B,C: probability for two computational-basis measurements.
    prob=np.real(np.diag(rho)).reshape(3,3,3)
    i,j=[SLOT[k] for k in pair];remainder=[v for v in range(3) if v not in (i,j)][0]
    return np.transpose(prob,(i,j,remainder)).sum(axis=2).reshape(9)

def exact_correct_prob(probs,pair,s):
    return sum(float(pr) for n,pr in enumerate(probs) if recover_symbol(tuple(pair),*divmod(n,3))==s)

def theoretical_correct(model,p):
    return {'x':1-p,'dephasing':1.0,'depolarizing':1-2*p/3,
            'erasure':1-p,'combined':(1-p)**2+p/3}[model]

def wilson_interval(k,n,z=1.95996398454):
    if n==0:return (float('nan'),float('nan'))
    p=k/n; z2=z*z; d=1+z2/n
    center=(p+z2/(2*n))/d
    half=z*(p*(1-p)/n+z2/(4*n*n))**.5/d
    return max(0,center-half),min(1,center+half)

# Theory/regression sanity checks, including several nonzero rates and all three pairs.
for model in MODELS:
    for s in SYMBOLS:
        for pair in PAIRS:
            v=VECTORS[s];slot=SLOT[pair[1]];pure=from_pure(v)
            for p in (0.,0.03,0.2,0.3):
                r=channel(pure,slot,p,model);quantum_validate(r)
                probs=pair_measure_probabilities(r,pair)
                assert np.isclose(probs.sum(),1,atol=2e-12)
                x=exact_correct_prob(probs,pair,s)*(1-p if model=='erasure' else 1)
                assert abs(x-theoretical_correct(model,p))<2e-12,(model,pair,s,p,x)
                if p==0: assert np.isclose(np.real(np.vdot(v,r@v)),1,atol=1e-12)
report('Analytical probability checks: PASS (X, dephasing, depolarizing, flagged erasure, combined, all pairs).')

## 5. Seeded Monte Carlo trials and full event-level raw results

In [ ]:
import time
beg=time.perf_counter()
events_path=OUT/'noise_trial_events.csv.gz'
summary_rows=[]
fidelity_rows=[]
with gzip.open(events_path,'wt',newline='',encoding='utf-8',compresslevel=6) as out:
    writer=csv.writer(out)
    writer.writerow(['config_id','model','pair','symbol','p','trial','seed','outcome','observed_symbol','expected_symbol',
                     'correct','wrong','aborted','diagnostic_mismatch'])
    for model in MODELS:
      for pair in PAIRS:
       for s in SYMBOLS:
        pure=from_pure(VECTORS[s]);slot=SLOT[pair[1]]
        for p in ERROR_RATES:
          cid=f'{model}_{pair}_s{s}_p{p:.2f}'
          seed=int.from_bytes(hashlib.sha256(f'{SEED}|{cid}'.encode()).digest()[:8],'big')
          rng=np.random.default_rng(seed)
          rho=channel(pure,slot,p,model)
          quantum_validate(rho)
          probs=pair_measure_probabilities(rho,pair)
          if model=='erasure':
              erasures=rng.random(N_SHOTS)<p
          else:erasures=np.zeros(N_SHOTS,dtype=bool)
          outcomes=rng.choice(9,size=N_SHOTS,p=probs/probs.sum())
          measurements=np.array([recover_symbol(tuple(pair),*divmod(int(o),3)) for o in outcomes],dtype=np.int8)
          ok=(measurements==s)&(~erasures)
          bad=(measurements!=s)&(~erasures)
          assert int(ok.sum()+bad.sum()+erasures.sum())==N_SHOTS
          for trial in range(N_SHOTS):
              aborted=bool(erasures[trial]);value='' if aborted else int(measurements[trial])
              writer.writerow([cid,model,pair,s,p,trial,seed,'ABORT' if aborted else ('CORRECT' if ok[trial] else 'WRONG'),
                               value,s,int(ok[trial]),int(bad[trial]),int(aborted),int(bad[trial])])
          correct=int(ok.sum());wrong=int(bad.sum());aborted=int(erasures.sum())
          theoretical=theoretical_correct(model,p)
          exact=exact_correct_prob(probs,pair,s)*(1-p if model=='erasure' else 1)
          fidelity=float(np.real(np.vdot(VECTORS[s],rho@VECTORS[s])))
          codespace=float(np.real(np.trace(CODE_PROJECTOR@rho)))
          if model=='erasure':
              conditional_fidelity=1.0
              orthogonal_erasure_fidelity=1-p
              fidelity_label='conditional_survivor_and_erasure_extended'
          else:
              conditional_fidelity=fidelity
              orthogonal_erasure_fidelity=float('nan')
              fidelity_label='unconditional_27d_density_matrix'
          assert -2e-12 <= fidelity <= 1+2e-12
          assert -2e-12 <= codespace <= 1+2e-12
          low,high=wilson_interval(correct,N_SHOTS)
          summary_rows.append(dict(model=model,pair=pair,symbol=s,p=p,shots=N_SHOTS,correct=correct,wrong=wrong,abort=aborted,
                correct_rate=correct/N_SHOTS,wrong_rate=wrong/N_SHOTS,abort_rate=aborted/N_SHOTS,
                wilson95_low=low,wilson95_high=high,exact_correct_probability=exact,
                theoretical_correct_probability=theoretical,diagnostic_known_s_mismatch=wrong,commitment_false_acceptance_tested=False,
                seed=seed,config_id=cid))
          fidelity_rows.append(dict(model=model,pair=pair,symbol=s,p=p,state_fidelity_27d=fidelity,
              conditional_surviving_fidelity=conditional_fidelity,
              erasure_extended_fidelity=orthogonal_erasure_fidelity,
              codespace_probability=codespace,interpretation=fidelity_label,config_id=cid))
summary_df=pd.DataFrame(summary_rows)
fidelity_df=pd.DataFrame(fidelity_rows)
summary_df.to_csv(OUT/'noise_trial_aggregates.csv',index=False)
fidelity_df.to_csv(OUT/'quantum_fidelity_codespace.csv',index=False)
expected_configs=len(MODELS)*len(PAIRS)*len(SYMBOLS)*len(ERROR_RATES)
assert len(summary_df)==expected_configs==315
assert int(summary_df.shots.sum())==expected_configs*N_SHOTS==315000
assert summary_df[['correct','wrong','abort']].sum().sum()==expected_configs*N_SHOTS
assert abs((summary_df.exact_correct_probability-summary_df.theoretical_correct_probability)).max()<2e-12
assert summary_df.loc[summary_df.p==0,'correct'].eq(N_SHOTS).all()
assert summary_df.loc[summary_df.model=='dephasing','correct'].eq(N_SHOTS).all()
assert summary_df.loc[summary_df.model=='dephasing','wrong'].eq(0).all()
assert summary_df.loc[summary_df.model=='erasure','wrong'].eq(0).all()
assert summary_df.loc[summary_df.model!='erasure','abort'].eq(0).all()
assert (~summary_df['commitment_false_acceptance_tested'].astype(bool)).all() # no production commitment test
elapsed=time.perf_counter()-beg
report('Executed:',len(summary_df),'configurations,',int(summary_df.shots.sum()),'trials,',round(elapsed,2),'seconds')
report('Per-channel total correct/wrong/abort:')
print(summary_df.groupby('model')[['correct','wrong','abort']].sum().to_string())

## 6. Historical Pauli-X comparison (new and original runs stay distinct)

In [ ]:
old=legacy_df.rename(columns={'x_error_p':'p','successes':'correct','success_rate':'correct_rate'})
old['run_label']='archived_quantum_2026_900shots_per_pair_p'
old['source_sha256']=LEGACY_X_SHA256
new=(summary_df[summary_df.model=='x'].groupby(['pair','p'],as_index=False)[['correct','wrong','shots']].sum())
new['correct_rate']=new.correct/new.shots
new['run_label']='stage03_1000shots_per_symbol_pair_p'
new['source_sha256']=digest(events_path.read_bytes())
comparison=pd.concat([old[['pair','p','shots','correct','correct_rate','run_label','source_sha256']],
                      new[['pair','p','shots','correct','correct_rate','run_label','source_sha256']]],ignore_index=True)
comparison.to_csv(OUT/'legacy_x_comparison.csv',index=False)
assert comparison.groupby('run_label').size().eq(21).all()
report('Historical X comparison:',len(comparison),'rows, no historical CSV modified.')

## 7. Figure generation and parameter sensitivity

The first figure explicitly keeps the archived 900-shot X series separate from the new 3,000-shot-per-pair series. Subsequent plots are generated from Stage 03 data. Wilson 95% confidence intervals are included in the CSV and shown as shaded bands on the noise-reconstruction figure. The exact density-matrix probabilities are also recorded independently.

In [ ]:
def save_plot(filename):
    plt.tight_layout();plt.savefig(OUT/filename,dpi=170,bbox_inches='tight');plt.close()

# Historical X comparison: aggregate each new set over its three symbols.
fig,ax=plt.subplots(figsize=(8,4.4))
colors={'AB':'tab:blue','AC':'tab:orange','BC':'tab:green'}
for pair in PAIRS:
    h=old[old.pair==pair].sort_values('p');n=new[new.pair==pair].sort_values('p')
    ax.plot(h.p,h.correct_rate,'--o',alpha=.65,color=colors[pair],label=f'{pair} legacy (900 shots/p)')
    ax.plot(n.p,n.correct_rate,'-s',color=colors[pair],label=f'{pair} Stage 03 (3000 shots/p)')
ax.plot(ERROR_RATES,[1-p for p in ERROR_RATES],color='black',lw=1.7,ls=':',label='Theory 1-p')
ax.set(xlabel='X-error probability p',ylabel='Correct classical-trit recovery probability',ylim=(0.63,1.025),title='QSV2G Pauli-X baseline: preserved historical and new results')
ax.grid(alpha=.25);ax.legend(fontsize=7,ncol=2);save_plot('legacy_vs_new_pauli_x.png')

agg=summary_df.groupby(['model','p'],as_index=False)[['correct','wrong','abort','shots']].sum()
agg['correct_rate']=agg.correct/agg.shots
agg['abort_rate']=agg.abort/agg.shots
agg['exact_correct_probability']=agg.apply(lambda r:theoretical_correct(r.model,r.p),axis=1)
intervals=np.array([wilson_interval(int(r.correct),int(r.shots)) for r in agg.itertuples()])
agg['wilson95_low']=intervals[:,0];agg['wilson95_high']=intervals[:,1]
agg.to_csv(OUT/'noise_channel_summary.csv',index=False)
fig,ax=plt.subplots(figsize=(8.4,4.5))
for model,g in agg.groupby('model',sort=False):
    ax.plot(g.p,g.correct_rate,marker='o',label=model)
    ax.fill_between(g.p.to_numpy(),g.wilson95_low.to_numpy(),g.wilson95_high.to_numpy(),alpha=0.12)
ax.set(xlabel='Channel parameter p',ylabel='Unconditional correct decoding probability',ylim=(0.5,1.02),title='QSV2G: classical recovery by simulated quantum noise')
ax.grid(alpha=.3);ax.legend();save_plot('noise_model_reconstruction.png')

fagg=fidelity_df.groupby(['model','p'],as_index=False)[['state_fidelity_27d','codespace_probability']].mean()
fig,ax=plt.subplots(figsize=(8.2,4.5))
for model,g in fagg.groupby('model',sort=False):
    if model=='erasure':continue # 27D conditional fidelity differs conceptually
    ax.plot(g.p,g.state_fidelity_27d,marker='o',label=model)
ax.set(xlabel='Channel parameter p',ylabel='Quantum encoded-state fidelity',ylim=(0.45,1.02),title='State fidelity versus classical recovery: different observables')
ax.grid(alpha=.3);ax.legend();save_plot('quantum_state_fidelity.png')

fig,ax=plt.subplots(figsize=(8.2,4.5))
for model in ('x','dephasing','depolarizing','combined'):
    g=fagg[fagg.model==model];ax.plot(g.p,g.codespace_probability,marker='o',label=model)
ax.set(xlabel='Channel parameter p',ylabel='Probability of ideal QSS code-space projection',ylim=(0.40,1.02),title='Code-space membership is not verification of logical intent')
ax.grid(alpha=.3);ax.legend();save_plot('qss_codespace_probability.png')

# The resource-level calculation is explicitly an independent-symbol theoretical extrapolation.
lengths=[1,9,27,81,162]
rr=[]
for model in MODELS:
 for p in ERROR_RATES:
  single=theoretical_correct(model,p)
  for L in lengths:
   rr.append({'model':model,'p':p,'L':L,'single_trit_theoretical_correct':single,
              'all_L_trits_correct_independent_model':single**L,
              'assumption':'independent identically distributed noise per trit; not a measured transaction success'})
rs=pd.DataFrame(rr);rs.to_csv(OUT/'resource_length_independence_extrapolation.csv',index=False)
fig,ax=plt.subplots(figsize=(8.1,4.5))
for L,g in rs[rs.model=='depolarizing'].groupby('L'):
    ax.plot(g.p,g.all_L_trits_correct_independent_model,marker='o',label=f'L={L}')
ax.set(xlabel='Depolarizing parameter p',ylabel='All L trits correct (iid model)',ylim=(-0.02,1.02),title='Theoretical full-resource success under independent errors')
ax.grid(alpha=.3);ax.legend();save_plot('resource_length_theoretical_reliability.png')
report('Saved: five publication-candidate figures and all underlying CSVs.')

## 8. Availability is not equivalent to classical workflow availability

In [ ]:
from itertools import combinations
ar=[]
for count in range(4):
 for offline in combinations('ABC',count):
  available=[x for x in 'ABC' if x not in offline]
  ar.append({'offline_authorities':''.join(offline) or '(none)','online_authorities':''.join(available),
             'qss_pair_exists':len(available)>=2,
             'qsv2g_full_workflow_possible_under_A_B_online_assumption':('A' in available and 'B' in available),
             'note':'Availability abstraction; no explicit quantum-channel erasure or real network modeled'})
avail=pd.DataFrame(ar)
avail.to_csv(OUT/'authority_availability_assumptions.csv',index=False)
assert avail[(avail.offline_authorities=='C')].iloc[0].qss_pair_exists
assert not avail[(avail.offline_authorities=='A')].iloc[0].qsv2g_full_workflow_possible_under_A_B_online_assumption
report('Availability: QSS threshold reconstruction and QSV2G full workflow distinguished.')

## 9. Independent statistical and artifact-integrity checks

In [ ]:
# Exact theory check and sampled deviations; stochastic variability should be reported, not optimized away.
row_count=len(summary_df)
totals=summary_df.groupby('model')[['shots','correct','wrong','abort']].sum().reset_index()
totals.to_csv(OUT/'outcome_totals_by_model.csv',index=False)
# Confidence interval coverage is not a pass/fail security guarantee; count whether finite-shot intervals contain exact p.
interval_contains=((summary_df.wilson95_low<=summary_df.theoretical_correct_probability) &
                  (summary_df.theoretical_correct_probability<=summary_df.wilson95_high)).sum()
# Monte Carlo sampling can produce outlier configurations: summarize, don't abort because of normal variation.
exact_diff=(summary_df.correct_rate-summary_df.exact_correct_probability).abs().max()
# Verify original model includes nontrivial separation between fidelity and classical recovery.
ph=fidelity_df[(fidelity_df.model=='dephasing') & (fidelity_df.p==0.30)]
assert (ph.state_fidelity_27d<0.999999).all()
assert (summary_df[(summary_df.model=='dephasing')&(summary_df.p==0.30)].correct_rate==1).all()
# Validate a fixed target share X-channel reduces recovery probability as expected.
assert theoretical_correct('x',0.3)==0.7
summary={'stage':'03','protocol':'QSV2G','result':'PASS_WITH_MODEL_LIMITATIONS',
  'channel_models':MODELS,'authority_pairs':PAIRS,'secret_symbols':SYMBOLS,
  'error_probabilities':ERROR_RATES,'seed_root':SEED,'shots_per_configuration':N_SHOTS,
  'configurations':row_count,'total_trials':int(summary_df.shots.sum()),
  'per_model_trial_counts':{str(a):int(b) for a,b in summary_df.groupby('model').shots.sum().items()},
  'theory_matches_exact_density_results':bool(abs(summary_df.exact_correct_probability-summary_df.theoretical_correct_probability).max()<2e-12),
  'wilson_intervals_cover_theoretical_count':int(interval_contains),
  'wilson_intervals_total':row_count,
  'max_finite_shot_absolute_deviation':float(exact_diff),
  'limitations':['Classical-trit reconstruction, not coherent reconstruction of arbitrary quantum input',
    'Local channels only on second selected share; no realistic quantum network',
    'Flagged erasure terminates selected pair; re-routing not executed',
    'Diagnostic mismatch versus known s is not malicious-verifiable QSS',
    'Resource length iid extrapolation is analytical and not observed end-to-end authorization',
    'Physical decoherence, multi-link loss, key handling, and transaction security are outside this stage'],
  'reference_core_sha256':CORE_SHA256,'reference_sectioniv_sha256':TEX_SHA256,
  'legacy_x_csv_sha256':LEGACY_X_SHA256,
  'run_environment':{'utc_generated':datetime.now(timezone.utc).isoformat(),'python':sys.version.splitlines()[0],
     'os':platform.platform(),'numpy':np.__version__,'pandas':pd.__version__,
     'matplotlib':matplotlib.__version__,'kaggle_path_detected':IN_KAGGLE},
  'source_note':'Reference core and Section IV embedded verbatim and SHA256-verified; model implementations are new Stage 03 code.'}
(OUT/'stage03_summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
report('Monte Carlo Wilson95 coverage:',int(interval_contains),'/',row_count,
       '| max abs difference from exact probability:',round(float(exact_diff),4))
report('STAGE RESULT:',summary['result'])

## 10. Output notes and archive export

**Citations and responsible interpretation:** Explain that dephasing may reduce phase coherence and encoded-state fidelity while leaving the classical recovery probability intact under computational-basis readout. A noisy-share value misdecoded under a trusted expected trit is a diagnostic, not a proof of malicious-dealer resistance. The `resource_length_independence_extrapolation.csv` is a **theoretical model**, not a measured transaction success rate. Cryptographic energy use, channel transport latency, authenticated entanglement distribution, and quantum-memory noise are not represented.

The notebook preserves historical X measurements unchanged and keeps its raw events (gzip CSV), aggregated metrics, fidelity, availability assumptions, five plots, source snapshots, environment metadata, and integrity manifest in one archive. **Provide the Kaggle-generated ZIP and saved executed `.ipynb`** to support traceability of that run.

In [ ]:
NOTES="""# QSV2G Stage 03 — Results and provenance

Quantum noise and classical trit recovery in an exact 27-dimensional three-qutrit density-matrix model.

## Experiment files
- noise_trial_events.csv.gz: 315,000 raw seeded trial events (including explicit erasure aborts).
- noise_trial_aggregates.csv: all 315 configurations, outcome counts, Wilson 95% intervals, analytic expectations.
- quantum_fidelity_codespace.csv: pure-state fidelity, code-space projection, erasure interpretation.
- noise_channel_summary.csv: model-by-p aggregates.
- legacy_qutrit_x_error_trials_original.csv: original 900-shot baseline (unaltered).
- legacy_x_comparison.csv: archive versus newly sampled X results; distinct runs and sizes.
- resource_length_independence_extrapolation.csv: iid mathematical extrapolation, NOT measured complete workflows.
- authority_availability_assumptions.csv: QSS threshold availability versus UC-A/UC-B workflow requirement.
- outcome_totals_by_model.csv: totals.
- five .png plots: measured reconstruction, quantum fidelity, code-space, legacy X, iid resource sensitivity.
- qsv2g_core_reference.py and QSV2G_SectionIV_reference.tex: full previous tested sources, SHA-256 verified.
- stage03_summary.json, stage03_artifact_manifest.json: environment and hashes.
- stage03_console_log.txt: notebook-written scientific progress/summary, NOT original Kaggle platform log.

## Methods
Channels act solely on the second share of the chosen pair. All models are ideal mathematical channels;
flagged share erasure aborts the selected pair attempt. Error p is a local channel parameter.

Recovery is classical computational-basis measurement of two shares. A dephasing channel can
lower fidelity while preserving computational-basis recovery. No formal security proof, real
quantum communication, physical authorization, or malicious-dealer resilience is established.

Previously archived X-error data have 900 trials per pair per p with random secret symbol.
The new Stage03 data have 1000 trials per symbol-pair-p, or 3000 per pair per p, and distinct seeds.

## What to upload
Send the produced ZIP and the Kaggle executed notebook, ideally with the saved Kaggle run URL/revision.
"""
(OUT/'README_RESULTS.md').write_text(NOTES,encoding='utf-8')
(OUT/'stage03_console_log.txt').write_text('\n'.join(LOG)+'\n',encoding='utf-8')
entries=[]
for p in sorted(OUT.rglob('*')):
    if p.is_file() and p.name!='stage03_artifact_manifest.json':
        raw=p.read_bytes()
        entries.append({'path':p.relative_to(OUT).as_posix(),'bytes':len(raw),'sha256':digest(raw)})
manifest={'stage':'03','hash_algorithm':'sha256','manifest_does_not_list_itself':True,
          'files':entries,'file_count':len(entries)}
(OUT/'stage03_artifact_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
with zipfile.ZipFile(ZIP_PATH,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=7) as z:
    for p in sorted(OUT.rglob('*')):
        if p.is_file():z.write(p,arcname=p.relative_to(OUT).as_posix())
with zipfile.ZipFile(ZIP_PATH) as z:
    assert z.testzip() is None
    mx=json.loads(z.read('stage03_artifact_manifest.json'))
    assert mx['file_count']==len(mx['files'])
    for f in mx['files']:
        data=z.read(f['path']);assert len(data)==f['bytes'] and digest(data)==f['sha256'],f['path']
report('STAGE03 ARCHIVE:',ZIP_PATH,'\nFILES:',len(entries)+1)
report('STAGE RESULT: PASS_WITH_MODEL_LIMITATIONS')